# INV-M365-D: Read vs Mutation Semantics

**Invariants proven:** INV-M365-D-001, INV-M365-D-002

**Lemmas referenced:** LEM-M365-D-001-01, LEM-M365-D-002-01

**Phase 1:** docs/math/instruction_contract.md (Sections 3, 4)

## 1. Setup

A_read and A_mut disjoint. A_mut ⊆ A_Admin. Spec-conformant eval: read-only -> tau'=tau, audit=[]; mutating only for Admin.

In [ ]:
P = {"Admin", "User"}
A_Admin, A_User = {"admin_mutate", "admin_read"}, {"user_read"}
A_read, A_mut = {"admin_read", "user_read"}, {"admin_mutate"}
assert A_read & A_mut == set()
assert A_mut <= A_Admin

def is_valid(instruction):
    if not isinstance(instruction, (tuple, list)) or len(instruction) != 4:
        return False
    p, a = instruction[0], instruction[2]
    return (p == "Admin" and a in A_Admin) or (p == "User" and a in A_User)

def eval_spec(instruction, tenant_state):
    if not is_valid(instruction):
        return None
    p, a = instruction[0], instruction[2]
    if a in A_read:
        return ("Success", tenant_state, [])
    if a in A_mut and p == "Admin":
        return ("Success", tenant_state + 1, [{}])
    return None

x_read = ("Admin", "i1", "admin_read", {})
x_mut = ("Admin", "i1", "admin_mutate", {})
tau = 0

## 2. Lemma Execution

D-001: Read-only -> new_tenant_state = tenant_state and |audit_events| = 0.
D-002: Mutating -> persona = Admin.

In [ ]:
r_read = eval_spec(x_read, tau)
r_mut = eval_spec(x_mut, tau)
assert r_read is not None and r_mut is not None
_, tau_read, audit_read = r_read
_, tau_mut, audit_mut = r_mut
assert tau_read == tau and len(audit_read) == 0, "D-001 lemma: read preserves state, no audit"
assert x_mut[0] == "Admin", "D-002 lemma: mutating instruction has Admin persona"

## 3. Invariant Verification

**INV-M365-D-001:** action in A_read and Eval defined -> new_tenant_state = tenant_state, |audit_events| = 0.

**INV-M365-D-002:** action in A_mut -> persona = Admin.

In [ ]:
def verify_D001(instruction, tenant_state, eval_fn):
    a = instruction[2]
    if a not in A_read:
        return
    out = eval_fn(instruction, tenant_state)
    if out is None:
        return
    r, new_state, audit = out
    assert new_state == tenant_state, "D-001: read preserves state"
    assert len(audit) == 0, "D-001: read produces no audit"

def verify_D002(instruction):
    a = instruction[2]
    if a in A_mut:
        assert instruction[0] == "Admin", "D-002: mutating requires Admin"

verify_D001(x_read, tau, eval_spec)
verify_D001(("User", "i1", "user_read", {}), tau, eval_spec)
verify_D002(x_mut)
print("INV-M365-D-001, D-002: pass_conditions hold.")

## 4. Failure Demonstration

User + mutating action -> invalid; eval returns None. So no defined outcome with persona != Admin for A_mut.

In [ ]:
x_bad = ("User", "i1", "admin_mutate", {})
assert eval_spec(x_bad, tau) is None
try:
    verify_D002(x_bad)
except AssertionError:
    pass
print("Failure case: User + admin_mutate correctly rejected.")

## 5. Conclusion

INV-M365-D-001 and INV-M365-D-002 hold under Phase 1 read/mutation semantics.

In [ ]:
print("VERIFY: INV-M365-D-001, INV-M365-D-002 — all pass.")